In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter
import scipy . ndimage . filters as filters
import pm

In [ ]:
fontanna_pow = cv2.imread('fontanna_pow.jpg')
fontanna_pow = cv2.cvtColor(fontanna_pow, cv2.COLOR_BGR2RGB)
fontanna1 = cv2.imread('fontanna1.jpg')
fontanna1 = cv2.cvtColor(fontanna1, cv2.COLOR_BGR2RGB)
fontanna2 = cv2.imread('fontanna2.jpg')
fontanna2 = cv2.cvtColor(fontanna2, cv2.COLOR_BGR2RGB)
budynek1 = cv2.imread('budynek1.jpg')
budynek1 = cv2.cvtColor(budynek1, cv2.COLOR_BGR2RGB)
budynek2 = cv2.imread('budynek2.jpg')
budynek2 = cv2.cvtColor(budynek2, cv2.COLOR_BGR2RGB)
eiffel1 = cv2.imread('eiffel1.jpg')
eiffel1 = cv2.cvtColor(eiffel1, cv2.COLOR_BGR2RGB)
eiffel2 = cv2.imread('eiffel2.jpg')
eiffel2 = cv2.cvtColor(eiffel2, cv2.COLOR_BGR2RGB)
left_panorama = cv2.imread('left_panorama.jpg')
left_panorama = cv2.cvtColor(left_panorama, cv2.COLOR_BGR2RGB)
right_panorama = cv2.imread('right_panorama.jpg')
right_panorama = cv2.cvtColor(right_panorama, cv2.COLOR_BGR2RGB)

# zadanie 1

In [ ]:
def calculate_H(image, filter_size=7):
    img_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    sobelx = cv2.Sobel(img_gray, cv2.CV_32F, 1, 0, ksize=filter_size)
    sobely = cv2.Sobel(img_gray, cv2.CV_32F, 0, 1, ksize=filter_size)

    Ixx = cv2.GaussianBlur(sobelx * sobelx, (filter_size, filter_size), 0)
    Iyy = cv2.GaussianBlur(sobely * sobely, (filter_size, filter_size), 0)
    Ixy = cv2.GaussianBlur(sobelx * sobely, (filter_size, filter_size), 0)

    K = 0.05
    H = np.zeros(img_gray.shape, dtype=np.float32)
    for y in range(img_gray.shape[0]):
        for x in range(img_gray.shape[1]):
            det = Ixx[y, x] * Iyy[y, x] - Ixy[y, x] * Ixy[y, x]
            trace = Ixx[y, x] + Iyy[y, x]
            H[y, x] = det - K * trace * trace

    H = (H - np.min(H)) / (np.max(H) - np.min(H))

    return H

In [ ]:
def find_max(image , size , threshold) : # size - maximum filter mask size
    data_max = filters.maximum_filter(image , size)
    maxima = (image == data_max)
    diff = image > threshold
    maxima[diff == 0] = 0
    return np.nonzero(maxima)

def draw_points(image, points):
    plt.figure()
    plt.imshow(image)
    plt.plot(points[1], points[0],'*', color='r')
    plt.show()

In [ ]:
fontana1_H = calculate_H(fontanna1, 7)
maxima1 = find_max(fontana1_H, 7, 0.55)
draw_points(fontanna1, maxima1)

fontanna2_H = calculate_H(fontanna2, 7)
maxima2 = find_max(fontanna2_H, 7, 0.55)
draw_points(fontanna2, maxima2)

In [ ]:
budynek1_H = calculate_H(budynek1, 7)
maxima3 = find_max(budynek1_H, 7, 0.55)
draw_points(budynek1, maxima3)

budynek2_H = calculate_H(budynek2, 7)
maxima4 = find_max(budynek2_H, 7, 0.55)
draw_points(budynek2, maxima4)

# zadanie 2

In [ ]:
def description(image, pts, size):
  Y, X, _ = image.shape
  img_grey = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
  pts = list(filter(lambda pt : pt[0] >= size and pt[0] < Y - size and pt[1]
                       >= size and pt[1] < X - size, zip (pts[0], pts[1])))
  filtered = []
  for pt in pts:
    patch = img_grey[pt[0] - size : pt[0] + size, pt[1] - size : pt[1] + size + 1]
    filtered.append(patch.flatten())
  return list(zip(filtered, pts))

In [ ]:
def description2(image, pts, size):
    Y, X, _ = image.shape
    img_grey = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    pts = list(filter(lambda pt: pt[0] >= size and pt[0] < Y - size and pt[1] >= size and pt[1] < X - size, zip(pts[0], pts[1])))
    filtered = []
    for pt in pts:
        patch = image[pt[0] - size: pt[0] + size + 1, pt[1] - size: pt[1] + size + 1]
        patch = patch.flatten()
        std = np.mean(patch)
        patch = (patch - std) / np.std(patch)
        filtered.append(patch)
    return list(zip(filtered, pts))

In [ ]:
def similarity(vec1, vec2):
    return sum(abs(vec1 - vec2))

In [ ]:
def compare_descriptions(desc1, desc2, n_best=1):
    matches = []
    for vec1, pt1 in desc1:
        min_val = float('inf')
        best = None
        for vec2, pt2 in desc2:
             val = similarity(vec1, vec2)
             if val < min_val:
                 min_val = val
                 best = [pt1, pt2, val]
        matches.append(best)
    matches.sort(key=lambda x: x[2])
    matches = matches[:n_best]
    return matches

In [ ]:
def zadanie2(img1, img2, size, threshold, n_best, version, grey = False):
    H1 = calculate_H(img1, size)
    pts1 = find_max(H1, size, threshold)
    if version == 1:
        desc1 = description(img1, pts1, size)
    else:
        desc1 = description2(img1, pts1, size)
    H2 = calculate_H(img2)
    pts2 = find_max(H2, size, threshold)
    if version == 1:
        desc2 = description(img2, pts2, size)
    else:
        desc2 = description2(img2, pts2, size)
    matches = compare_descriptions(desc1, desc2, n_best)
    if grey:
        img1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
        img2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
    pm.plot_matches(img1, img2, matches)

In [ ]:
zadanie2(fontanna1, fontanna2, 15, 0.55, 20, 1, True)

In [ ]:
zadanie2(fontanna1, fontanna2, 15, 0.55, 20, 2, True)

In [ ]:
zadanie2(budynek1, budynek2, 5, 0.5, 20, 1, True)

In [ ]:
zadanie2(budynek1, budynek2, 5, 0.5, 20, 2, True)

In [ ]:
zadanie2(eiffel1, eiffel2, 5, 0.5, 20, 1, True)

In [ ]:
zadanie2(eiffel1, eiffel2, 5, 0.5, 20, 2, True)

# zadanie 3

In [ ]:
def corner_score(intensities, center_intensity, threshold, n=9):
    brighter = 0
    darker = 0

    for intensity in intensities:
        intensity= int(intensity)
        center_intensity= int(center_intensity)
        if intensity > center_intensity + threshold:
            brighter += 1
            if brighter >= n:
                return True
        elif intensity < center_intensity - threshold:
            darker += 1
            if darker >= n:
                return True

    return False

In [ ]:
def fast_detector(image, threshold, n=9, window_size=3):
    harr = calculate_H(image, 7)

    img_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    height, width = img_gray.shape[:2]

    points = []
    harr_values = []

    for y in range(window_size, height - window_size):
        for x in range(window_size, width - window_size):
            center_intensity = img_gray[y, x]
            intensities = [
                img_gray[y - 3, x + 1],
                img_gray[y - 3, x],
                img_gray[y - 3, x - 1],
                img_gray[y - 2, x + 2],
                img_gray[y - 2, x - 2],
                img_gray[y - 1, x + 3],
                img_gray[y - 1, x - 3],
                img_gray[y, x + 3],
                img_gray[y, x - 3],
                img_gray[y + 1, x + 3],
                img_gray[y + 1, x - 3],
                img_gray[y + 2, x + 2],
                img_gray[y + 2, x - 2],
                img_gray[y + 3, x + 1],
                img_gray[y + 3, x],
                img_gray[y + 3, x - 1]
            ]

            if corner_score(intensities, center_intensity, threshold, n):
                points.append((y, x))
                harr_values.append(harr[y, x])

    return points, harr_values

In [ ]:
def detect_fast_points(image, threshold=40, n_points=200):
    points, harr_values = fast_detector(image, threshold)

    harris_points = list(zip(points, harr_values))
    harris_points.sort(key=lambda x: x[1], reverse=True)
    harris_points = harris_points[:n_points]

    return {key: value for key, value in harris_points}

In [ ]:
def non_max_suppression(image, points: dict):

    values_to_remove = []

    for point, value in points.items():
        if (point, value) in values_to_remove:
            continue

        y, x = point
        values = []
        for i in range(y - 1, y + 2):
            for j in range(x - 1, x + 2):
                if i < 0 or j < 0 or i >= image.shape[0] - 1 or j >= image.shape[1] - 1:
                    continue
                if (i, j) in points:
                    values.append(((i, j), points[(i, j)]))

        if len(values) > 1:
            values.sort(key=lambda x: x[1])
            for v in values[:-1]:
                if v not in values_to_remove:
                    values_to_remove.append(v)

    for v in values_to_remove:
        points.pop(v[0])

    return points

In [ ]:
def filter_out_too_small_surrounding(image, filtered_points, size=16):
    y, x = image.shape[:2]

    points = list(filter(lambda pt: pt[0][0] >= size and pt[0][0] < y - size
                         and pt[0][1] >= size and pt[0][1] < x - size, list(zip(filtered_points.keys(), filtered_points.values()))))

    return points

In [ ]:
def chose_n_best_points(filtered_points, n=30):
    if len(filtered_points) <= n:
        return filtered_points
    filtered_points.sort(key=lambda x: x[1], reverse=True)
    filtered_points = filtered_points[:n]
    return filtered_points

In [ ]:
def centroids(image, points, size=9):
    img_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    centroids = []
    thetas = []

    for point, value in points:
        y, x = point
        m00 = 0
        m01 = 0
        m10 = 0
        for i in range(y - size, y + size + 1):
            for j in range(x - size, x + size + 1):
                dy = y - i
                dx = x - j
                distance2 = dx * dx + dy * dy
                if distance2 <= size * size:
                    m00 += int(img_gray[i, j])
                    m01 += -dy * int(img_gray[i, j])
                    m10 += -dx * int(img_gray[i, j])
        C = (m10 / m00, m01 / m00)
        theta = np.arctan2(m01, m10)
        centroids.append(C)
        thetas.append(theta)

    return centroids, thetas

In [ ]:
def load_brief(image_patch, theta):
    score = []
    with open('orb_descriptor_positions.txt', 'r') as f:
        for line in f:
            values = line.split(' ')
            values = [float(x) for x in values]

            u = values[:2]
            v = values[2:]

            u[0] = int(np.cos(theta)*u[0] + np.sin(theta)*u[1])
            u[1] = int(np.sin(theta)*u[1] + np.cos(theta)*u[0])

            v[0] = int(np.cos(theta)*v[0] + np.sin(theta)*v[1])
            v[1] = int(np.sin(theta)*v[1] + np.cos(theta)*v[0])

            u = tuple(u)
            v = tuple(v)

            if image_patch[u] < image_patch[v]:
                score.append(1)
            else:
                score.append(0)

    return score

In [ ]:
def describe_points_brief(image, points, thetas, size=15):
    img_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    scores = []
    points = [x[0] for x in points]
    for i, point in enumerate(points):
        y, x = point
        theta = thetas[i]
        patch = img_gray[y - size: y + size + 1, x - size: x + size + 1]
        patch_gauss = cv2.GaussianBlur(patch, (5, 5), 0)
        brief_scorescore = load_brief(patch_gauss, theta)
        scores.append(brief_scorescore)

    return scores

In [ ]:
def hamming(arr1, arr2):
    val = 0
    for i in range(len(arr1)):
        if i >= len(arr2):
            break
        if arr1[i] != arr2[i]:
            val += 1
    return val

def compare_brief_descriptions(scores1, points1, scores2, points2, n=30):
    comparison = []

    for p1, s1 in zip(points1, scores1):
        min_val = float('inf')
        best_val = None
        for p2, s2 in zip(points2, scores2):
            diff = hamming(s1, s2)
            if diff < min_val:
                min_val = diff
                best_val = [p1, p2, diff]

        comparison.append(best_val)

    comparison.sort(key=lambda x: x[2])
    comparison = comparison[:n]

    return comparison

In [ ]:
def orb(image1, threshold=50, n_best=20):
    points1 = detect_fast_points(image1, threshold)
    points1 = non_max_suppression(image1, points1)
    points1 = filter_out_too_small_surrounding(image1, points1)
    points1 = chose_n_best_points(points1, n_best)
    centroids1, thetas1 = centroids(image1, points1)
    scores1 = describe_points_brief(image1, points1, thetas1)
    points1 = [x[0] for x in points1]

    return scores1, points1

In [ ]:
def two_image_orb(image1, image2, threshold=50, n_best=20, gray=False):
    scores1, points1 = orb(image1, threshold, n_best)
    scores2, points2 = orb(image2, threshold, n_best)

    comparison = compare_brief_descriptions(scores1, points1, scores2, points2)

    if gray:
        image1 = cv2.cvtColor(image1, cv2.COLOR_RGB2GRAY)
        image2 = cv2.cvtColor(image2, cv2.COLOR_RGB2GRAY)

    pm.plot_matches(image1, image2, comparison)

In [ ]:
def scaled_images(img, scale, n=3):
    imgs = [img]
    for i in range(n):
        img = cv2.resize(img, (0, 0), fx=1/scale, fy=1/scale)
        imgs.append(img)
    return imgs

def scale_point(pt, scale):
    return (pt[0] * scale, pt[1] * scale)

In [ ]:
def two_image_orb_pyramid(image1, image2, scale=2, gray=False):
    scaled_images_1 = scaled_images(image1, scale)
    scaled_images_2 = scaled_images(image2, scale)

    scores_1_s, points_1_s, scores_2_s, points_2_s = [], [], [], []

    for i in range(len(scaled_images_1)):
        scores1, points1 = orb(scaled_images_1[i])
        scores2, points2 = orb(scaled_images_2[i])

        points1 = [x * 2**i for x in points1]
        points2 = [x * 2**i for x in points2]

        for j in range(len(scores1)):
            scores_1_s.append(scores1[j])
            points_1_s.append(scale_point(points1[j], scale**i))

        for j in range(len(scores2)):
            scores_2_s.append(scores2[j])
            points_2_s.append(scale_point(points2[j], scale**i))

    comparison = compare_brief_descriptions(scores_1_s, points_1_s, scores_2_s, points_2_s)

    if gray:
        image1 = cv2.cvtColor(image1, cv2.COLOR_RGB2GRAY)
        image2 = cv2.cvtColor(image2, cv2.COLOR_RGB2GRAY)

    pm.plot_matches(image1, image2, comparison)

In [ ]:
two_image_orb(fontanna1, fontanna2)
two_image_orb_pyramid(fontanna1, fontanna2)

In [ ]:
two_image_orb(budynek1, budynek2)
two_image_orb_pyramid(budynek1, budynek2)

In [ ]:
two_image_orb(eiffel1, eiffel2, gray=True)
two_image_orb_pyramid(eiffel1, eiffel2, gray=True)

# zadanie 4

In [ ]:
left_panorama_g = cv2.cvtColor(left_panorama, cv2.COLOR_BGR2GRAY)
right_panorama_g = cv2.cvtColor(right_panorama, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp1_L, des1 = sift.detectAndCompute(left_panorama_g, None)
kp2_R, des2 = sift.detectAndCompute(right_panorama_g, None)

In [ ]:
kp1 = cv2.drawKeypoints(left_panorama_g, kp1_L, None, color=(255, 0, 0))
kp2 = cv2.drawKeypoints(right_panorama_g, kp2_R, None, color=(255, 0, 0))

In [ ]:
plt.subplot(1, 2, 1)
plt.imshow(kp1)
plt.subplot(1, 2, 2)
plt.imshow(kp2)
plt.show()

In [ ]:
matcher = cv2.BFMatcher(cv2.NORM_L2)
matches = matcher.knnMatch(des1, des2, k=2)

best_matches = []
for m, n in matches:
    if m.distance < 0.5 * n.distance:
       best_matches.append(m)

In [ ]:
im = cv2.drawMatches(left_panorama_g, kp1_L, right_panorama_g, kp2_R, best_matches, None, matchesThickness=1)
plt.imshow(im)
plt.show()

In [ ]:
ptsA = np.float32([kp1_L[m.queryIdx].pt for m in best_matches])
ptsB = np.float32([kp2_R[m.trainIdx].pt for m in best_matches])

H, status = cv2.findHomography(ptsA, ptsB, cv2.RANSAC, 5.0)

height, width = right_panorama_g.shape
result = cv2.warpPerspective(left_panorama_g, H, (width + left_panorama_g.shape[1], height))

result[0:right_panorama_g.shape[0], 0:right_panorama_g.shape[1]] = right_panorama_g

plt.figure(figsize=(15, 10))
plt.imshow(result, cmap='gray')
plt.axis('off')
plt.title('Panorama Before Cropping')
plt.show()

_, thresh = cv2.threshold(result, 1, 255, cv2.THRESH_BINARY)
coords = np.column_stack(np.where(thresh > 0))
x, y, w, h = cv2.boundingRect(coords)

margin_x = int(w * 0.4)
margin_y = int(h * 0.4)

x = max(x - margin_x, 0)
y = max(y - margin_y, 0)
w = min(w + 2 * margin_x, result.shape[1] - x)
h = min(h + 2 * margin_y, result.shape[0] - y)

cropped_result = result[y:y + h, x:x + w]

plt.figure(figsize=(15, 10))
plt.imshow(cropped_result, cmap='gray')
plt.axis('off')
plt.title('Final Cropped Panorama')
plt.show()

In [ ]:
plt.subplot(1, 3, 1)
plt.imshow(left_panorama)
plt.subplot(1, 3, 2)
plt.imshow(right_panorama)
plt.subplot(1, 3, 3)
plt.imshow(result)
plt.show()